# 08 — Feature Ablation Analysis

Which feature groups matter for B1 (global RF) and B3 (per-creator RF)?

**Design rules (enforced):**
- All model selection uses CV on train only — test is touched once at the end.
- No engagement_daily features from the current video.
- All time-dependent features use data as-of before publish.
- For B3: vs_creator group is excluded (always), and near-constant within-creator features are naturally ignored by RF.

**Pre-requisite:** run `python scripts/run_ablation.py` first (~50 min).

In [ ]:
import matplotlib; matplotlib.use('Agg')
import os, sys, time, warnings
from pathlib import Path

if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir(os.path.dirname(os.getcwd()))
sys.path.insert(0, 'src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
import joblib
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

from creator_analytics import features as fe
from creator_analytics.config import load_config
from creator_analytics.models import VS_CREATOR_SUFFIX, CreatorModel, drop_constant_features
from creator_analytics.evaluate import compute_metrics, compute_per_creator_metrics, predict_per_creator
from creator_analytics.ablation import build_group_map, build_combinations, filter_combinations_b3

cfg       = load_config('config/default.yaml')
PROCESSED = Path(cfg['paths']['processed_dir'])
PREDS_DIR = Path('results/predictions')
FIGS      = Path('results/figures'); FIGS.mkdir(parents=True, exist_ok=True)
SEED      = cfg['seed']
print('Imports OK')

In [ ]:
data, spec = fe.load_feature_table(PROCESSED)
idx = pd.read_parquet(PROCESSED / 'dataset_index.parquet')

FEATURES_GLOBAL = spec['features']
FEATURES_PC     = [f for f in FEATURES_GLOBAL if not f.endswith(VS_CREATOR_SUFFIX)]

TRAIN = idx['split'].eq('train')
TEST  = idx['split'].eq('test')

X_train_global = data.loc[TRAIN.values, FEATURES_GLOBAL].reset_index(drop=True)
X_train_pc     = data.loc[TRAIN.values, FEATURES_PC].reset_index(drop=True)
y_train        = idx.loc[TRAIN, 'target_rel'].reset_index(drop=True)
auth_train     = idx.loc[TRAIN, 'author_id'].reset_index(drop=True)
X_test_global  = data.loc[TEST.values, FEATURES_GLOBAL].reset_index(drop=True)
X_test_pc      = data.loc[TEST.values, FEATURES_PC].reset_index(drop=True)
auth_test      = idx.loc[TEST, 'author_id'].reset_index(drop=True)

nb04 = pd.read_parquet(PREDS_DIR / 'nb04_predictions.parquet')
y_test = nb04['y_true']

GROUPS    = build_group_map(spec)
combos    = build_combinations(FEATURES_GLOBAL, FEATURES_PC, GROUPS)
combos_b3 = filter_combinations_b3(combos)
combos_map    = {c['combination']: c for c in combos}
combos_b3_map = {c['combination']: c for c in combos_b3}

print(f'Train: {len(X_train_global):,}  Test: {len(X_test_global):,}  Creators: {auth_train.nunique()}')
print(f'Global features: {len(FEATURES_GLOBAL)}  PC features: {len(FEATURES_PC)}')
print(f'Combinations: B1={len(combos)}, B3={len(combos_b3)}')

In [ ]:
print('Feature groups (cat_* merged):')
print(f'{"Group":22s}  {"N":>4}  Features (first 6...)')
total = 0
for g, feats in GROUPS.items():
    preview = ', '.join(feats[:6]) + ('…' if len(feats) > 6 else '')
    print(f'  {g:20s}  {len(feats):4d}  {preview}')
    total += len(feats)
print(f'  {"TOTAL":20s}  {total:4d}')

In [ ]:
ABLATION_CSV = Path('results/ablation.csv')
if not ABLATION_CSV.exists():
    raise FileNotFoundError(
        f'{ABLATION_CSV} not found. Run: python scripts/run_ablation.py'
    )
df = pd.read_csv(ABLATION_CSV)
print(f'Loaded {len(df)} rows from {ABLATION_CSV}')
print(df[['model', 'combination', 'n_features', 'cv_rmse_mean', 'cv_rmse_std',
          'cv_spearman_mean', 'rmse_vs_baseline']].to_string(index=False))

In [ ]:
# Leave-one-out: RMSE increase = cv_rmse(LOO) - cv_rmse(all_features)
# Positive = group is useful (removing it hurts)

def loo_deltas(df_model):
    loo = df_model[df_model['type'] == 'leave_one_out'].copy()
    loo['group'] = loo['combination'].str.replace('LOO_', '', regex=False)
    loo['delta_rmse']    = loo['rmse_vs_baseline']
    loo['delta_spearman'] = loo['spearman_vs_baseline']
    return loo.sort_values('delta_rmse', ascending=False)

df_b1 = df[df['model'] == 'B1_RF_global']
df_b3 = df[df['model'] == 'B3_RF_pc']
loo_b1 = loo_deltas(df_b1)
loo_b3 = loo_deltas(df_b3)

# Sort both by B1 order for easy comparison
group_order = loo_b1['group'].tolist()
loo_b3_sorted = loo_b3.set_index('group').reindex(
    [g for g in group_order if g in loo_b3.set_index('group').index]
).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=False)

for ax, loo_df, title in [
    (axes[0], loo_b1, 'B1 — Global RF (106 feats)'),
    (axes[1], loo_b3_sorted, 'B3 — Per-creator RF (99 feats)'),
]:
    colors = ['#d73027' if v > 0 else '#4575b4' for v in loo_df['delta_rmse']]
    ax.barh(loo_df['group'], loo_df['delta_rmse'], color=colors)
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_xlabel('RMSE increase when group excluded (positive = group is useful)')
    ax.set_title(title)
    ax.invert_yaxis()
    for bar, val in zip(ax.patches, loo_df['delta_rmse']):
        ax.text(
            val + (0.0003 if val >= 0 else -0.0003),
            bar.get_y() + bar.get_height() / 2,
            f'{val:+.4f}', va='center', ha='left' if val >= 0 else 'right', fontsize=8
        )

plt.suptitle('Leave-one-group-out: group contribution to model performance', fontsize=13)
plt.tight_layout()
plt.savefig(FIGS / 'nb08_loo_contribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved nb08_loo_contribution.png')

In [ ]:
# Single-group: what does each group contribute on its own?
def single_summary(df_model, model_label):
    sg = df_model[df_model['type'] == 'single_group'].copy()
    sg['group'] = sg['combination'].str.replace('SINGLE_', '', regex=False)
    baseline_rmse = df_model[df_model['combination'] == 'all_features']['cv_rmse_mean'].iloc[0]
    sg = sg.sort_values('cv_rmse_mean')
    print(f'\n{model_label} — Single-group CV RMSE (lower = group contains more signal):')
    print(f'  Baseline (all features): {baseline_rmse:.4f}')
    print(f'  {"Group":22s}  {"N feats":>7}  {"RMSE":>8}  {"vs baseline":>12}')
    for _, row in sg.iterrows():
        print(f'  {row["group"]:22s}  {row["n_features"]:7d}  '
              f'{row["cv_rmse_mean"]:8.4f}  {row["rmse_vs_baseline"]:+12.4f}')

single_summary(df_b1, 'B1 Global RF')
single_summary(df_b3, 'B3 Per-creator RF')

In [ ]:
# Meaningful combinations summary
for model_label, df_model in [('B1 Global RF', df_b1), ('B3 Per-creator RF', df_b3)]:
    mc = df_model[df_model['type'] == 'meaningful'].copy()
    mc = mc.sort_values('cv_rmse_mean')
    print(f'\n{model_label} — Meaningful combinations:')
    print(f'  {"Combination":28s}  {"N feats":>7}  {"CV RMSE":>9}  '
          f'{"±std":>7}  {"vs baseline":>12}  {"CV Spearman":>12}')
    for _, row in mc.iterrows():
        print(f'  {row["combination"]:28s}  {row["n_features"]:7d}  '
              f'{row["cv_rmse_mean"]:9.4f}  {row["cv_rmse_std"]:7.4f}  '
              f'{row["rmse_vs_baseline"]:+12.4f}  {row["cv_spearman_mean"]:12.4f}')

In [ ]:
# Test evaluation: baseline + best-by-CV combo for each model (ONCE, no further tuning)
B1_PARAMS_ABLATION = {
    'n_estimators': 100, 'min_samples_split': 5,
    'min_samples_leaf': 5, 'max_features': 0.3, 'max_depth': None,
}
B3_CONFIG = {
    'n_estimators': 100, 'max_depth': 4,
    'min_samples_leaf': 5, 'max_features': 0.4,
}

def _train_b3_rf(X_tr_sub, y_tr_sub, auth_tr_sub, rf_cfg, seed):
    """Train per-creator RF only (no XGB overhead). Returns dict[author_id → CreatorModel]."""
    auth_tr_sub = auth_tr_sub.reset_index(drop=True)
    X_tr_sub    = X_tr_sub.reset_index(drop=True)
    y_tr_sub    = y_tr_sub.reset_index(drop=True)

    def _fit(a):
        mask = auth_tr_sub == a
        Xa = X_tr_sub[mask].reset_index(drop=True)
        ya = y_tr_sub[mask].reset_index(drop=True)
        if len(Xa) < 5:
            return None
        X_clean, _ = drop_constant_features(Xa, threshold=1.0)
        if X_clean.shape[1] == 0:
            return None
        pipe = Pipeline([
            ('imp', SimpleImputer(strategy='median')),
            ('rf',  RandomForestRegressor(**rf_cfg, n_jobs=1, random_state=seed)),
        ])
        pipe.fit(X_clean, ya)
        return CreatorModel(
            author_id=a, model=pipe,
            feature_names=list(X_clean.columns),
            cv_rmse=float('nan'), model_name='RF',
            n_train=len(Xa), n_features=X_clean.shape[1],
            worse_than_zero=False,
        )

    raw = joblib.Parallel(n_jobs=-1, verbose=0)(
        joblib.delayed(_fit)(a) for a in auth_tr_sub.unique()
    )
    return {r.author_id: r for r in raw if r is not None}


ABLATION_CSV = Path('results/ablation.csv')
df_test_rows = []

for model_name, df_model, feat_col, X_tr, X_te in [
    ('B1_RF_global', df_b1, 'features_global', X_train_global, X_test_global),
    ('B3_RF_pc',     df_b3, 'features_pc',     X_train_pc,     X_test_pc),
]:
    baseline_row = df_model[df_model['combination'] == 'all_features'].iloc[0]
    best_row     = df_model.sort_values('cv_rmse_mean').iloc[0]

    print(f'\n{"="*60}')
    print(f'{model_name}')
    print(f'  Baseline CV RMSE : {baseline_row["cv_rmse_mean"]:.4f} ± {baseline_row["cv_rmse_std"]:.4f}')
    print(f'  Best CV combo    : {best_row["combination"]} '
          f'RMSE={best_row["cv_rmse_mean"]:.4f} ± {best_row["cv_rmse_std"]:.4f}')
    delta_cv = best_row['cv_rmse_mean'] - baseline_row['cv_rmse_mean']
    print(f'  Delta (best−baseline): {delta_cv:+.4f}')
    if delta_cv < -1e-4 and best_row['combination'] != 'all_features':
        print(f'  *** Best CV combo beats all_features. Consider updating B1/B3 — see conclusions.')

    to_eval = [('all_features', baseline_row)]
    if best_row['combination'] != 'all_features':
        to_eval.append((best_row['combination'], best_row))

    for combo_name, row in to_eval:
        combo_def = combos_map.get(combo_name) or combos_b3_map.get(combo_name, {})
        feats = combo_def[feat_col]

        t0 = time.time()
        if model_name == 'B1_RF_global':
            pipe = Pipeline([
                ('imp', SimpleImputer(strategy='median')),
                ('rf',  RandomForestRegressor(**B1_PARAMS_ABLATION, n_jobs=-1, random_state=SEED)),
            ])
            pipe.fit(X_tr[feats], y_train)
            preds = pipe.predict(X_te[feats])
        else:
            rf_models = _train_b3_rf(X_tr[feats], y_train, auth_train, B3_CONFIG, SEED)
            preds = predict_per_creator(
                X_te[feats].reset_index(drop=True),
                auth_test.reset_index(drop=True),
                rf_models,
            )
            preds = np.where(np.isnan(preds), float(y_train.mean()), preds)

        m   = compute_metrics(y_test, preds)
        pc  = compute_per_creator_metrics(
            y_test,
            pd.Series(preds, index=y_test.index),
            auth_test.reset_index(drop=True),
        )
        med_pc = float(pc['spearman'].median())
        print(f'  {combo_name:28s}  test RMSE={m["rmse"]:.4f}  '
              f'Spear={m["spearman"]:.4f}  medPC={med_pc:.4f}  ({time.time()-t0:.0f}s)')
        df_test_rows.append({
            'model': model_name, 'combination': combo_name,
            'test_rmse': m['rmse'], 'test_mae': m['mae'],
            'test_spearman': m['spearman'],
            'test_median_pc_spearman': med_pc,
        })

# Update ablation.csv with test results
df_updated = df.copy()
for row in df_test_rows:
    mask = (df_updated['model'] == row['model']) & (df_updated['combination'] == row['combination'])
    for col in ['test_rmse', 'test_mae', 'test_spearman', 'test_median_pc_spearman']:
        df_updated.loc[mask, col] = row[col]
df_updated.to_csv(ABLATION_CSV, index=False)
print(f'\nUpdated {ABLATION_CSV} with test results.')

In [ ]:
print('=== CONCLUSIONS — Feature Ablation ===')
print()

for model_label, df_model, feat_col in [
    ('B1 Global RF', df_b1, 'n_features'),
    ('B3 Per-creator RF', df_b3, 'n_features'),
]:
    baseline = df_model[df_model['combination'] == 'all_features'].iloc[0]
    best     = df_model.sort_values('cv_rmse_mean').iloc[0]
    worst_loo = df_model[df_model['type'] == 'leave_one_out'].sort_values(
        'rmse_vs_baseline', ascending=False).iloc[0]
    best_loo  = df_model[df_model['type'] == 'leave_one_out'].sort_values(
        'rmse_vs_baseline').iloc[0]
    content = df_model[df_model['combination'] == 'content_only']
    context = df_model[df_model['combination'] == 'context_only']

    print(f'── {model_label} ──')
    print(f'  Baseline (all features): CV RMSE = {baseline["cv_rmse_mean"]:.4f}')
    print(f'  Most important group  : {worst_loo["groups_label"]} '
          f'(+{worst_loo["rmse_vs_baseline"]:.4f} without it)')
    print(f'  Least important group : {best_loo["groups_label"]} '
          f'({best_loo["rmse_vs_baseline"]:+.4f} without it)')
    if not content.empty:
        print(f'  Content-only RMSE     : {content.iloc[0]["cv_rmse_mean"]:.4f} '
              f'({content.iloc[0]["rmse_vs_baseline"]:+.4f} vs baseline)')
    if not context.empty:
        print(f'  Context-only RMSE     : {context.iloc[0]["cv_rmse_mean"]:.4f} '
              f'({context.iloc[0]["rmse_vs_baseline"]:+.4f} vs baseline)')
    if best['combination'] != 'all_features':
        delta = best['cv_rmse_mean'] - baseline['cv_rmse_mean']
        print(f'  Best CV combo         : {best["combination"]} '
              f'({delta:+.4f} vs baseline) — consider updating B1/B3')
    else:
        print(f'  Best CV combo         : all_features (adding features does not hurt)')
    print()

# Global vs per-creator comparison
print('── Global vs Per-creator differences ──')
cc_b1 = df_b1[df_b1['combination'] == 'LOO_creator_context']['rmse_vs_baseline']
cc_b3 = df_b3[df_b3['combination'] == 'LOO_creator_context']['rmse_vs_baseline'] if 'LOO_creator_context' in df_b3['combination'].values else pd.Series([float('nan')])
if not cc_b1.empty:
    print(f'  creator_context removal: B1 {cc_b1.iloc[0]:+.4f}  B3 {cc_b3.iloc[0] if not cc_b3.empty else "n/a"}'
          f'  (expected: low for B3 — constant within creator)')
vc_b1 = df_b1[df_b1['combination'] == 'without_vs_creator']['rmse_vs_baseline']
if not vc_b1.empty:
    print(f'  vs_creator removal (B1 only): {vc_b1.iloc[0]:+.4f}'
          f'  (positive = vs_creator features add value for global model)')